# SpaceX Falcon 9 First Stage Landing Prediction

## Lab 1: Collecting the Data — SpaceX REST API

In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each. Much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch.

In this lab, we will make a get request to the SpaceX API, do some basic data wrangling, and export the cleaned data for the next labs.

## Objectives

- Make a GET request to the SpaceX REST API (`/v4/launches/past`) to retrieve past launch records
- Use helper functions to enrich the data with booster, payload, launchpad and core sub-data referenced by IDs
- Build a pandas DataFrame, filter it to Falcon 9 launches only, and deal with missing values
- Export the result to `dataset_part_1.csv`

In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

### Helper functions

The launch records only contain IDs referencing other API endpoints (rockets, payloads, launchpads, cores). The following helper functions take the ID column of a launch row, call the corresponding endpoint, and extract the attributes we need: booster version, launch site, payload mass/orbit, and core landing details.

In [ ]:
def getBoosterVersion(data):
    """Takes the dataset and uses the rocket column to call the API and append the booster name."""
    for x in data['rocket']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/rockets/" + str(x)).json()
            BoosterVersion.append(response['name'])

In [ ]:
def getLaunchSite(data):
    """Takes the dataset and uses the launchpad column to call the API and append launch site info."""
    for x in data['launchpad']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/launchpads/" + str(x)).json()
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])

In [ ]:
def getPayloadData(data):
    """Takes the dataset and uses the payloads column to call the API and append payload mass and orbit."""
    for load in data['payloads']:
        if load:
            response = requests.get("https://api.spacexdata.com/v4/payloads/" + load).json()
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])

In [ ]:
def getCoreData(data):
    """Takes the dataset and uses the cores column to call the API and append core landing details."""
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get("https://api.spacexdata.com/v4/cores/" + core['core']).json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

### Request and parse SpaceX launch data

We now request the past launch data from the SpaceX API. The response is a JSON array of launch objects, which we normalize into a flat pandas DataFrame.

In [ ]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print(response.status_code)

In [ ]:
data = pd.json_normalize(response.json())
data.head()

You will notice that a lot of the data are IDs. For example, the `rocket` column has no information about the rocket, just an identification number. We now use the API again to get information about the launches using the IDs given for each launch. Specifically, we will be using the columns `rocket`, `payloads`, `launchpad`, and `cores`.

In [ ]:
# Take a subset of the dataframe keeping only the features we want and the flight number / date.
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# Remove rows with multiple cores (Falcon Heavy) or multiple payloads.
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# Since payloads and cores are lists of size 1, extract the single feature and replace it.
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# Convert the date_utc to a datetime datatype and then extract the date.
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Restrict to launches before 2020-11-13.
data = data[data['date'] <= datetime.date(2020, 11, 13)]
data.head()

### Enrich the data using the helper functions

We initialize global lists for every feature, then call the helper functions. Each function loops over the rows, calls the corresponding API endpoint with the row's ID, and appends the extracted values to the lists. Finally we assemble the lists into a dictionary and build the final DataFrame.

In [ ]:
# Global variables
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

# Call the helper functions
getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)

In [ ]:
launch_dict = {'FlightNumber': list(data['flight_number']),
               'Date': list(data['date']),
               'BoosterVersion': BoosterVersion,
               'PayloadMass': PayloadMass,
               'Orbit': Orbit,
               'LaunchSite': LaunchSite,
               'Outcome': Outcome,
               'Flights': Flights,
               'GridFins': GridFins,
               'Reused': Reused,
               'Legs': Legs,
               'LandingPad': LandingPad,
               'Block': Block,
               'ReusedCount': ReusedCount,
               'Serial': Serial,
               'Longitude': Longitude,
               'Latitude': Latitude}

df = pd.DataFrame(launch_dict)
df.head()

### Filter the DataFrame to Falcon 9 launches

The API returns Falcon 1, Falcon 9 and Falcon Heavy launches. Since we only care about Falcon 9, we filter the DataFrame to keep only rows whose booster version is Falcon 9, then reset the FlightNumber column.

In [ ]:
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].copy()
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
data_falcon9.head()

### Data Wrangling — dealing with missing values

We calculate the percentage of missing values in each attribute. The `PayloadMass` column has a few missing values; since it is a numeric feature, we replace the missing values with the mean of the column. `LandingPad` is missing whenever the booster did not attempt a pad landing, so we keep those NaNs for now.

In [ ]:
data_falcon9.isnull().sum() / data_falcon9.shape[0] * 100

In [ ]:
payload_mean = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(payload_mean)
data_falcon9.isnull().sum() / data_falcon9.shape[0] * 100

### Export the data

The cleaned dataset is exported to `dataset_part_1.csv`. In the next labs we will perform exploratory data analysis and data wrangling on this file. The overall flow of this notebook was: **GET request to the SpaceX API -> normalize JSON -> resolve IDs via helper functions -> build DataFrame -> filter Falcon 9 -> impute missing payload mass -> export CSV**.

In [ ]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)